# Caesium in hydrogen

Caesium evaporated into a negative-ion source is ionized by the plasma electrons
and scatters on the H₂ gas. `plasmacoll.caesium` holds these collisions: the
electrons are a background with $n_e(x)$ and $T_e(x)$, and an ionization turns a
Cs marker into a Cs⁺ of the same weight; the new electron is not tracked.

In [ ]:
import plasma_plots as pp

pp.set_backend("plotly")  # interactive figures, embedded in the docs
pp.set_math("latex")  # formulas in the figures typeset as LaTeX

In [ ]:
import cunumpy as xp
import numpy as np

import plasmacoll
from plasmacoll.caesium import H2_MASS
from plasmacoll.constants import ELECTRON_MASS

sigma = plasmacoll.caesium_ionization_cross_section()
energy = np.geomspace(3.0, 1000.0, 400)
with pp.figure(title="e + Cs → Cs⁺ + 2e (Lotz, 6s)") as fig:
    ax = fig[0]
    ax.loglog(energy, xp.to_numpy(sigma(xp.asarray(energy))))
    ax.set_xlabel("electron energy [eV]")
    ax.set_ylabel(r"cross section $\sigma$ [m$^2$]")
    ax.set_ylim(1e-21, 3e-19)

## Rate and ionization length

The rate coefficient $K(T_e) = \langle \sigma v \rangle$ over a Maxwellian sets the
ionization frequency $n_e K$. A Cs atom evaporated at 0.1 eV (about 380 m/s) travels
$v / (n_e K)$ before it is ionized.

In [ ]:
temperature = np.geomspace(0.5, 20.0, 60)
rate = plasmacoll.rate_coefficient(sigma, temperature, ELECTRON_MASS)
speed = 380.0
with pp.figure(title="Ionization length of caesium") as fig:
    ax = fig[0]
    for density in (1e16, 1e17, 1e18):
        ax.loglog(
            temperature, speed / (density * rate), label=f"$n_e$ = {density:.0e} m⁻³"
        )
    ax.set_xlabel("electron temperature $T_e$ [eV]")
    ax.set_ylabel("ionization length $\\lambda$ [m]")
    ax.legend()

## Markers

`caesium_collisions` builds the collision operator for `Cs` and `Cs⁺` markers on
the electron background and the H₂ gas. Markers at rest in a plasma with
$n_e = 10^{17}$ m⁻³ and $T_e = 2$ eV ionize at $n_e K$:

In [ ]:
electrons = plasmacoll.electron_background(1e17, 2.0)
gas = plasmacoll.NeutralBackground("H2", 1e19, 0.03, H2_MASS)
mcc = plasmacoll.caesium_collisions(electrons, gas, seed=1)
num, dt = 1_000_000, 2e-7
result = mcc.collide("Cs", xp.zeros((num, 1)), xp.zeros((num, 3)), xp.ones(num), dt)
k = float(plasmacoll.rate_coefficient(sigma, 2.0, ELECTRON_MASS))
print("ionized:", result.diagnostics.counts["ionization:Cs"])
print("expected:", round(num * 1e17 * k * dt))

The Cs + H₂ cross section is a hard-sphere placeholder and the Lotz formula
leaves out the 5p shell; see the module docstring for the sources.